In [59]:
import os
import glob
import numpy as np
import scipy.io as sio
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import random

In [60]:
from scipy.io import loadmat
data = loadmat('/kaggle/input/competitions/ioai-computer-vision-challenge-2026/labeled/labeled/ground-truth/GT_IMG_1.mat')

In [61]:
IMG_DIR = '/kaggle/input/competitions/ioai-computer-vision-challenge-2026/labeled/labeled/images'
GT_DIR = '/kaggle/input/competitions/ioai-computer-vision-challenge-2026/labeled/labeled/ground-truth'
TEST_DIR = '/kaggle/input/competitions/ioai-computer-vision-challenge-2026/test/test'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_points(gt_path):
    mat = sio.loadmat(gt_path)
    return np.array(mat['image_info'][0][0][0][0][0]).reshape(-1, 2).astype(np.float32)

def make_density(points, h, w, sigma):
    dm = np.zeros((h, w), dtype=np.float32)
    for x, y in points:
        xi, yi = int(x), int(y)
        if 0 <= yi < h and 0 <= xi < w:
            dm[yi, xi] += 1.0
    return ndimage.gaussian_filter(dm, sigma=sigma, mode='constant')


In [62]:
class CrowdDataset(Dataset):
    def __init__(self, img_paths, gt_paths, train=True, sigma=4, crop=384):
        self.img_paths = img_paths
        self.gt_paths = gt_paths
        self.train = train
        self.sigma = sigma
        self.crop = crop

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        points = load_points(self.gt_paths[idx])
        w, h = img.size

        if self.train:
            ch = max(64, min(self.crop, h) // 8 * 8)
            cw = max(64, min(self.crop, w) // 8 * 8)
            x0 = random.randint(0, w - cw)
            y0 = random.randint(0, h - ch)
            img = img.crop((x0, y0, x0 + cw, y0 + ch))
            points = points - np.array([x0, y0], dtype=np.float32)
            mask = (points[:, 0] >= 0) & (points[:, 0] < cw) & (points[:, 1] >= 0) & (points[:, 1] < ch)
            points = points[mask]
            if random.random() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if len(points):
                    points[:, 0] = cw - 1 - points[:, 0]
            h, w = ch, cw
        else:
            w8, h8 = max(64, w // 8 * 8), max(64, h // 8 * 8)
            img = img.crop((0, 0, w8, h8))
            mask = (points[:, 0] < w8) & (points[:, 1] < h8)
            points = points[mask]
            h, w = h8, w8

        dm = make_density(points, h, w, self.sigma)
        dm = torch.from_numpy(dm).unsqueeze(0).unsqueeze(0)
        dm = nn.functional.adaptive_avg_pool2d(dm, (h // 8, w // 8)) * 64
        return NORM(img), dm.squeeze(0)


In [63]:
class CSRNet(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        self.frontend = nn.Sequential(*list(vgg.features.children())[:23])
        self.backend = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(512, 256, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=2, dilation=2), nn.ReLU(inplace=True),
        )
        self.output_layer = nn.Conv2d(64, 1, 1)
        for m in list(self.backend.modules()) + [self.output_layer]:
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.output_layer(self.backend(self.frontend(x)))


In [64]:
def train(epochs=20):
    imgs = sorted(glob.glob(os.path.join(IMG_DIR, '*')))
    gts = sorted(glob.glob(os.path.join(GT_DIR, '*')))
    n_val = max(1, len(imgs) // 10)
    train_ds = CrowdDataset(imgs[:-n_val], gts[:-n_val], train=True)
    val_ds = CrowdDataset(imgs[-n_val:], gts[-n_val:], train=False)
    train_dl = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

    model = CSRNet().to(DEVICE)
    criterion = nn.MSELoss(reduction='sum')
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

    best_mae = float('inf')
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for inputs, targets in train_dl:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        mae, rmse = 0.0, 0.0
        with torch.no_grad():
            for inputs, targets in val_dl:
                pred = model(inputs.to(DEVICE)).sum().item()
                gt = targets.sum().item()
                mae += abs(pred - gt)
                rmse += (pred - gt) ** 2
        mae /= len(val_ds)
        rmse = (rmse / len(val_ds)) ** 0.5

        if mae < best_mae:
            best_mae = mae
            torch.save(model.state_dict(), 'csrnet_best.pth')

        print(f"Epoch {epoch+1}: loss={epoch_loss/len(train_ds):.4f} val_MAE={mae:.2f} val_RMSE={rmse:.2f} best_MAE={best_mae:.2f}")

train()


Epoch 1: loss=3.2974 val_MAE=63.10 val_RMSE=80.17 best_MAE=63.10
Epoch 2: loss=2.2017 val_MAE=63.56 val_RMSE=77.61 best_MAE=63.10
Epoch 3: loss=3.0425 val_MAE=20.80 val_RMSE=30.51 best_MAE=20.80
Epoch 4: loss=2.6164 val_MAE=29.24 val_RMSE=39.65 best_MAE=20.80
Epoch 5: loss=1.6676 val_MAE=26.73 val_RMSE=37.34 best_MAE=20.80
Epoch 6: loss=2.6892 val_MAE=99.78 val_RMSE=112.75 best_MAE=20.80
Epoch 7: loss=4.4978 val_MAE=49.72 val_RMSE=55.08 best_MAE=20.80
Epoch 8: loss=2.3822 val_MAE=52.74 val_RMSE=67.03 best_MAE=20.80
Epoch 9: loss=1.9616 val_MAE=28.48 val_RMSE=37.61 best_MAE=20.80
Epoch 10: loss=2.6385 val_MAE=13.58 val_RMSE=19.42 best_MAE=13.58
Epoch 11: loss=3.1309 val_MAE=62.67 val_RMSE=70.14 best_MAE=13.58
Epoch 12: loss=2.2879 val_MAE=25.72 val_RMSE=32.65 best_MAE=13.58
Epoch 13: loss=2.0978 val_MAE=45.10 val_RMSE=50.86 best_MAE=13.58
Epoch 14: loss=1.8164 val_MAE=27.98 val_RMSE=37.35 best_MAE=13.58
Epoch 15: loss=2.8975 val_MAE=25.26 val_RMSE=28.11 best_MAE=13.58
Epoch 16: loss=2.3

In [65]:
def inference():
    model = CSRNet().to(DEVICE)
    model.load_state_dict(torch.load('csrnet_best.pth', map_location=DEVICE))
    model.eval()

    test_images = sorted(glob.glob(os.path.join(TEST_DIR, '*')))
    results = []

    with torch.no_grad():
        for img_path in tqdm(test_images):
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            img = img.crop((0, 0, max(64, w // 8 * 8), max(64, h // 8 * 8)))
            img_tensor = NORM(img).unsqueeze(0).to(DEVICE)
            count = model(img_tensor).sum().item()
            filename = os.path.basename(img_path)
            results.append({'image_name': filename[-11:], 'count': max(0.0, count)})

    pd.DataFrame(results).to_csv('submission.csv', index=False)

inference()

100%|██████████| 216/216 [00:33<00:00,  6.42it/s]


In [66]:
from tqdm import tqdm
def inference():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = CSRNet().to(device)
    model.load_state_dict(torch.load('csrnet_model.pth', map_location=device))
    model.eval()
    
    test_dir = '/kaggle/input/competitions/ioai-computer-vision-challenge-2026/test/test'
    test_images = sorted(glob.glob(os.path.join(test_dir, '*')))
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    results = []
    
    with torch.no_grad():
        for img_path in tqdm(test_images):
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(device)
            
            output = model(img_tensor)
            count = output.sum().item()
            
            filename = os.path.basename(img_path)
            results.append({'image_name': filename[-11:], 'count': count})
            
    df = pd.DataFrame(results)
    df.to_csv('submission.csv', index=False)


In [67]:
%%time
inference()

100%|██████████| 216/216 [00:33<00:00,  6.44it/s]

CPU times: user 38.6 s, sys: 701 ms, total: 39.3 s
Wall time: 35.1 s
